In [30]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../datasets").resolve()
df = pd.read_csv(DATA_DIR / "air_quality.csv")

df.head()

,datetime_utc,datetime_local,date,hour,day_of_week,month,year,location_id,location_name,latitude,longitude,sensor_type,pm10,pm25
0,2020-04-09 17:00:00+00:00,2020-04-09 23:00:00+05:00,2020-04-09,23,3,4,2020,8876,Almaty,43.252855,76.93118,unknown,NaN,6.0
1,2020-04-09 18:00:00+00:00,2020-04-10 00:00:00+05:00,2020-04-10,0,4,4,2020,8876,Almaty,43.252855,76.93118,unknown,NaN,10.0
2,2020-04-09 19:00:00+00:00,2020-04-10 01:00:00+05:00,2020-04-10,1,4,4,2020,8876,Almaty,43.252855,76.93118,unknown,NaN,8.0
3,2020-04-09 20:00:00+00:00,2020-04-10 02:00:00+05:00,2020-04-10,2,4,4,2020,8876,Almaty,43.252855,76.93118,unknown,NaN,6.0
4,2020-04-09 21:00:00+00:00,2020-04-10 03:00:00+05:00,2020-04-10,3,4,4,2020,8876,Almaty,43.252855,76.93118,unknown,NaN,4.0


In [31]:
df = df.drop(columns=['datetime_utc', 'datetime_local', 'date', 'location_name', 'sensor_type', 'pm10', 'day_of_week'])
df.head()

,hour,month,year,location_id,latitude,longitude,pm25
0,23,4,2020,8876,43.252855,76.93118,6.0
1,0,4,2020,8876,43.252855,76.93118,10.0
2,1,4,2020,8876,43.252855,76.93118,8.0
3,2,4,2020,8876,43.252855,76.93118,6.0
4,3,4,2020,8876,43.252855,76.93118,4.0


In [32]:
def convert_hours(hour):
    if hour <= 5:
        return 'Night'
    elif hour <= 11:
        return 'Morning'
    elif hour <= 17:
        return 'Afternoon'
    elif hour <= 23:
        return 'Evening'
    
def convert_seasons(month):
    if month in [1, 2, 12]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Fall'

In [33]:
df['TimeOfDay'] = df['hour'].apply(convert_hours)
df['Season'] = df['month'].apply(convert_seasons)
df = df.drop(columns=['hour', 'month'])
df.head()

,year,location_id,latitude,longitude,pm25,TimeOfDay,Season
0,2020,8876,43.252855,76.93118,6.0,Evening,Spring
1,2020,8876,43.252855,76.93118,10.0,Night,Spring
2,2020,8876,43.252855,76.93118,8.0,Night,Spring
3,2020,8876,43.252855,76.93118,6.0,Night,Spring
4,2020,8876,43.252855,76.93118,4.0,Night,Spring


In [39]:
if 'Season' not in df.columns:
    df['Season'] = df['month'].apply(convert_seasons)

if 'TimeOfDay' not in df.columns:
    df['TimeOfDay'] = df['hour'].apply(convert_hours)

location_coords = df[['location_id', 'latitude', 'longitude']].drop_duplicates()

agg = (
    df
    .groupby(['location_id', 'year', 'Season', 'TimeOfDay'], as_index=False)['pm25']
    .mean()
    .rename(columns={'pm25': 'mean_pm25'})
)

years = list(range(2020, 2027))
locations = df['location_id'].unique()
seasons = ['Winter', 'Spring', 'Summer', 'Fall']
times = ['Night', 'Morning', 'Afternoon', 'Evening']

idx = pd.MultiIndex.from_product([locations, years, seasons, times],
                                 names=['location_id', 'year', 'Season', 'TimeOfDay'])

full = (
    agg
    .set_index(['location_id', 'year', 'Season', 'TimeOfDay'])
    .reindex(idx)
    .reset_index()
)

full = full.merge(location_coords, on='location_id', how='left')


pivot = full.pivot_table(index=['location_id', 'latitude', 'longitude', 'year'],
                         columns=['Season', 'TimeOfDay'],
                         values='mean_pm25')

full.head(20)
full.to_csv(DATA_DIR / 'air_quality_agg.csv', index=False)
# pivot.head(30)